In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def systematic_resample(mypacman, myghost, p, g):

    for turn in range(T):
        print('\nTurn #{}'.format(turn+1))

        # real pacman movement
        mypacman = mypacman.move(0.2, 10.0)

        # real move for ghost (random)
        gturn = (random.random()-0.5)*(math.pi/2.0)
        gdist = random.random()*20.0
        myghost = myghost.move(gturn, gdist)

        # move particles
        p2 = []
        g2 = []
        for i in range(N):
            p2.append(p[i].move(0.2, 10.0))
            g2.append(g[i].move(gturn, gdist))
        p = p2
        g = g2

        show_belief(mypacman, p, myghost, g)
        print("Average pacman distance before resample:", eval(mypacman, p))
        print("Average ghost distance before resample:", eval(myghost, g))

        # observe
        ZP = mypacman.sense()
        ZG = myghost.sense(mypacman)

        ###################################
        ###### YOUR CODE STARTS HERE ######
        ###################################

        # weight particles
        w_pacman = []
        for pac in p:
            prob = pac.measurement_prob(ZP)
            w_pacman.append(prob)

        w_ghost = []
        for ghst in g:
            prob = ghst.measurement_prob(ZG, mypacman)
            w_ghost.append(prob)

        # systematic sampling Pacman
        w_sum = sum(w_pacman)
        w_pacman = [w / w_sum for w in w_pacman]

        cumulative_sum = np.cumsum(w_pacman)
        positions = (np.arange(N) + random.random()) / N
        indexes = np.zeros(N, dtype=int)

        i = 0
        j = 0
        while i < N:
            if positions[i] < cumulative_sum[j]:
                indexes[i] = j
                i += 1
            else:
                j += 1

        p2 = []
        for k in range(N):
            p2.append(p[indexes[k]])
        p = p2

        # systematic sampling Ghost
        w_sum_ghost = sum(w_ghost)
        w_ghost = [w / w_sum_ghost for w in w_ghost]

        cumulative_sum_ghost = np.cumsum(w_ghost)
        positions_ghost = (np.arange(N) + random.random()) / N
        indexes_ghost = np.zeros(N, dtype=int)

        i = 0
        j = 0
        while i < N:
            if positions_ghost[i] < cumulative_sum_ghost[j]:
                indexes_ghost[i] = j
                i += 1
            else:
                j += 1

        g2 = []
        for k in range(N):
            g2.append(g[indexes_ghost[k]])
        g = g2

        #################################
        ###### YOUR CODE ENDS HERE ######
        #################################

        show_belief(mypacman, p, myghost, g)
        print("Average pacman distance after resample:", eval(mypacman, p))
        print("Average ghost distance after resample:", eval(myghost, g))

        pacman_dist = eval(mypacman, p)
        ghost_dist = eval(myghost, g)


    return p, g, pacman_dist, ghost_dist

In [ ]:
def run_particle_filter(mypacman, myghost, p, g):

    for turn in range(T):
        print('\nTurn #{}'.format(turn+1))

        # real pacman movement
        mypacman = mypacman.move(0.2, 10.0)

        # real move for ghost (random)
        gturn = (random.random()-0.5)*(math.pi/2.0)
        gdist = random.random()*20.0
        myghost = myghost.move(gturn, gdist)

        # move particles
        p2 = []
        g2 = []
        for i in range(N):
            p2.append(p[i].move(0.2, 10.0))
            g2.append(g[i].move(gturn, gdist))
        p = p2
        g = g2

        show_belief(mypacman, p, myghost, g)
        print("Average pacman distance before resample:", eval(mypacman, p))
        print("Average ghost distance before resample:", eval(myghost, g))

        # observe
        ZP = mypacman.sense()
        ZG = myghost.sense(mypacman)

        ###################################
        ###### YOUR CODE STARTS HERE ######
        ###################################

        # weight particles
        w_pacman = []
        for pac in p:
            prob = pac.measurement_prob(ZP)
            w_pacman.append(prob)

        w_ghost = []
        for ghst in g:
            prob = ghst.measurement_prob(ZG, mypacman)
            w_ghost.append(prob)

        # resample using spinning wheel (Pacman)
        p2 = []
        index = int(random.random()*N)
        beta = 0
        mw = max(w_pacman)

        for i in range(N):
            beta += random.random() * 2 * mw
            while beta > w_pacman[index]:
                beta -= w_pacman[index]
                index = (index + 1) % N
            p2.append(p[index])
        p = p2

        # resample ghost particles
        g2 = []
        index = int(random.random()*N)
        beta = 0
        mw = max(w_ghost)

        for i in range(N):
            beta += random.random() * 2 * mw
            while beta > w_ghost[index]:
                beta -= w_ghost[index]
                index = (index + 1) % N
            g2.append(g[index])
        g = g2

        #################################
        ###### YOUR CODE ENDS HERE ######
        #################################

        show_belief(mypacman, p, myghost, g)
        print("Average pacman distance after resample:", eval(mypacman, p))
        print("Average ghost distance after resample:", eval(myghost, g))

        pacman_dist = eval(mypacman, p)
        ghost_dist = eval(myghost, g)


    return p, g, pacman_dist, ghost_dist

In [ ]:
saved_particles = []

for run in range(10):

    # pacman/ghost initialization
    forward_noise = 3.0 
    turn_noise = 0.05
    sense_noise = 3.0

    mypacman = pacman()
    mypacman.set_noise(forward_noise, turn_noise, sense_noise)

    myghost = ghost()
    myghost.set_noise(forward_noise, turn_noise, sense_noise)

    # particle distribution
    N = 1000
    T = 10

    p = []
    g = []

    for i in range(N):
        x = pacman()
        x.set_noise(forward_noise, turn_noise, sense_noise)
        p.append(x)

        x = ghost()
        x.set_noise(forward_noise, turn_noise, sense_noise)
        g.append(x)

    # save the particle sets
    saved_particles.append((mypacman, myghost, p.copy(), g.copy()))

In [ ]:

wheel = []

# compare methods
for x in range(10):

    myp, myg, p0, g0 = saved_particles[x]
 
    p, g, pacman_dist, ghost_dist = run_particle_filter(myp, myg, p0, g0)

    wheel.append((pacman_dist, ghost_dist))


In [ ]:
pac_results = np.mean([x[0] for x in wheel[1:10]])


ghost_results = np.mean([x[1] for x in wheel[1:10]])

print("Average pacman distance over 10 runs:", pac_results)
print("Average ghost distance over 10 runs:", ghost_results)    

In [ ]:

systematic = []

# compare methods
for x in range(10):

    myp, myg, p0, g0 = saved_particles[x]
 
    p, g, pacman_dist, ghost_dist = systematic_resample(myp, myg, p0, g0)

    systematic.append((pacman_dist, ghost_dist))


In [ ]:
pac_results_systematic = np.mean([x[0] for x in systematic[1:10]])


ghost_results_systematic = np.mean([x[1] for x in systematic[1:10]])

print("Average pacman distance over 10 runs:", pac_results_systematic)
print("Average ghost distance over 10 runs:", ghost_results_systematic)    